# 02 · Reaction-class distributions

Class counts overall, by month, by direction, and by post-market
collection mode — plus coverage and insufficient-data rates. All of this
is aggregation of stored feature rows; no reaction is reclassified here.

> **This notebook is a research client, not a pipeline.** It contains no SQL, no
> threshold, and no classification rule. Every number comes from
> `afterhours_lab.research`, which reads persisted features computed once by
> `afterhours_lab.reactions`. Nothing here writes to the database — the pool is
> opened read-only.

In [ ]:
import datetime as dt

from afterhours_lab.research import (
    EventFilter,
    fetch_cohort,
    fetch_event_detail,
    fetch_class_distribution,
    fetch_monthly_counts,
    to_csv,
    to_jsonl,
    to_pandas,
    to_polars,
    write_parquet,
)
from afterhours_lab.research.notebook import (
    research_pool,
    describe_filter,
    describe_cohort,
    show_cohort,
    development_split,
)

# Jupyter already runs an event loop, so `await` works at cell top level.
pool = await research_pool()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
today = dt.date.today()
cohort_filter = EventFilter(
    date_from=today - dt.timedelta(days=365),
    date_to=today,
    limit=5000,
)
async with pool.acquire() as conn:
    cohort = show_cohort(await fetch_cohort(conn, cohort_filter))
    distribution = await fetch_class_distribution(conn, cohort_filter)
    monthly = await fetch_monthly_counts(conn, cohort_filter)

## Class counts over the whole cohort

In [ ]:
labels = [c for c, _ in distribution]
counts = [n for _, n in distribution]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels, counts)
ax.set_xlabel('events')
ax.set_title('reaction class (persisted features)')
ax.invert_yaxis()
plt.tight_layout()

## Analyzed vs total, by month

The gap between the two lines is the operations backlog, not a market
observation — keep it visually separate from the class mix.

In [ ]:
months = [m for m, _, _ in monthly]
analyzed = [a for _, a, _ in monthly]
total = [t for _, _, t in monthly]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(months, total, label='in universe')
ax.bar(months, analyzed, label='analyzed (complete)')
ax.set_ylabel('events')
ax.set_title('monthly coverage of the analysis')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

## By direction and by collection mode

Straight `groupby` on the stored columns.

In [ ]:
df = to_pandas(cohort.rows)
analyzed_df = df[df['analysis_status'] == 'complete']
print('direction split:')
print(analyzed_df['reaction_direction'].value_counts(dropna=False))
print()
print('post-market collection mode:')
print(analyzed_df['earnings_postmarket_collection_mode'].value_counts(dropna=False))

In [ ]:
pivot = analyzed_df.pivot_table(
    index='reaction_class',
    columns='reaction_direction',
    values='symbol',
    aggfunc='count',
    fill_value=0,
)
pivot

## Coverage and insufficient-data rates

In [ ]:
status_counts = df['analysis_status'].value_counts(dropna=False)
print(status_counts)
print()
n = len(df)
if n:
    insufficient = int(status_counts.get('insufficient_data', 0))
    not_analyzed = int(df['analysis_status'].isna().sum())
    print(f'insufficient_data: {insufficient / n:.1%} of the universe')
    print(f'not analyzed yet:  {not_analyzed / n:.1%} of the universe')

In [ ]:
study_ready = df['covered_phases'].fillna('').str.contains('earnings_postmarket')
print(f'events with the post-market study phase captured: {study_ready.mean():.1%}')

In [ ]:
await pool.close()